In [73]:
import numpy as np
import json
import random
from collections import deque
from pathlib import Path

In [81]:
GRID_SIZE = 5
NUM_GRIDS = 100

random.seed(42)
np.random.seed(42)

In [82]:
def is_goal_reachable(obstacles, start, goal):
    queue = deque([start])
    visited = {start}

    directions = [
        (-1, 0),  # North
        (1, 0),   # South
        (0, -1),  # West
        (0, 1)    # East
    ]

    while queue:
        r, c = queue.popleft()

        if (r, c) == goal:
            return True

        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            next_pos = (nr, nc)

            if (
                0 <= nr < GRID_SIZE
                and 0 <= nc < GRID_SIZE
                and next_pos not in obstacles
                and next_pos not in visited
            ):
                visited.add(next_pos)
                queue.append(next_pos)

    return False


In [83]:
def generate_grids(num_grids, grid_size):
    grids = []
    seen = set()

    while len(grids) < num_grids:
        start_pos = (random.randint(0, grid_size - 1), random.randint(0, grid_size - 1))
        goal_pos = (random.randint(0, grid_size - 1), random.randint(0, grid_size - 1))
        while goal_pos == start_pos:
            goal_pos = (random.randint(0, grid_size - 1), random.randint(0, grid_size - 1))

        
        open_positions = [
            (r, c)
            for r in range(grid_size)
            for c in range(grid_size)
            if (r, c) not in [start_pos, goal_pos]
        ]


        num_obstacles = random.randint(4, 6)
        obstacles = frozenset(
            random.sample(open_positions, num_obstacles)
        )

        # Unique obstacle configuration
        if obstacles in seen:
            continue

        # Only retain grids with reachable goals
        if is_goal_reachable(obstacles, start_pos, goal_pos):
          seen.add(obstacles)

          grid = np.full((grid_size, grid_size), '.')

          grid[start_pos] = 'S'  # Start
          grid[goal_pos] = 'G'  # Goal

          for r, c in obstacles:
              grid[r, c] = 'B'

          grids.append({
            'id': len(grids),
            'grid': grid.tolist(),
            'start': start_pos,
            'goal': goal_pos
          })

    return grids

In [84]:
grids = generate_grids(NUM_GRIDS, GRID_SIZE)

print("Number of grids:", len(grids))

Number of grids: 100


In [85]:
print(grids[99]['grid'])

print("Start position:", grids[99]['start'])
print("Goal position:", grids[99]['goal'])

[['B', '.', '.', '.', '.'], ['.', '.', '.', '.', 'B'], ['.', '.', '.', 'S', 'G'], ['.', '.', 'B', '.', '.'], ['B', '.', '.', 'B', '.']]
Start position: (2, 3)
Goal position: (2, 4)


In [86]:
print("Current directory:", Path.cwd())

with open("data/grids.json", "w") as f:
    json.dump(grids, f, indent=2)

Current directory: /home/sm/projects/llm-grid-solver
